# 06 Batch. Homework

In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName('06-homework') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 15:32:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In this homework we'll put what we learned about Spark in practice.

For this homework we will be using the Yellow 2025-11 data from the official website:

```bash
wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
```

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [3]:
!ls -la

total 82184
drwxr-xr-x 1 spark spark      192 Mar  4 15:23 .
drwxr-xr-x 1 root  root        52 Mar  4 15:30 ..
-rw-r--r-- 1 spark spark     4128 Mar  4 15:32 homework.ipynb
-rw-r--r-- 1 spark spark     3196 Mar  4 15:12 homework.md
drwxr-xr-x 1 spark spark       96 Mar  4 15:23 output
-rw-r--r-- 1 spark spark 71134255 Dec 19 15:51 yellow_tripdata_2025-11.parquet


## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

* `4.1.1`

> [!NOTE]
> To install PySpark follow this [guide](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/06-batch/setup/pyspark.md)

In [ ]:
spark.version

## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB [x]
- 75MB
- 100MB

In [4]:
df_yellow = spark.read.option("recursiveFileLookup", "true").parquet('./yellow_tripdata_2025-11.parquet')

In [ ]:
df_yellow \
    .repartition(4) \
    .write.parquet('output/yellow', mode='overwrite')

In [ ]:
!ls -la --b=M  output/yellow | grep ".snappy.parquet$"

## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610
- 102,340
- 162,604 [x]
- 225,768

In [5]:
from pyspark.sql import functions as F

In [6]:
df_yellow= df_yellow.withColumn('pickup_date', F.to_date(df_yellow.tpep_pickup_datetime)).withColumn('dropoff_date', F.to_date(df_yellow.tpep_dropoff_datetime))
df_yellow.show()
     

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_date|dropoff_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|  

In [10]:
df_yellow.filter((df_yellow.pickup_date == '2025-11-15')).count()

162604

## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 22.7
- 58.2
- 90.6 [x]
- 134.5

In [14]:
df_yellow = df_yellow.withColumn(
    "trip_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600.0
)

df_yellow.select(F.max("trip_hours").alias("max_trip_hours")).show()

+-----------------+
|   max_trip_hours|
+-----------------+
|90.64666666666666|
+-----------------+



## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040 [x]
- 8080

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

- Governor's Island/Ellis Island/Liberty Island
- Arden Heights [x]
- Rikers Island
- Jamaica Bay

If multiple answers are correct, select any

In [15]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-04 15:43:31--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 99.84.149.117, 99.84.149.165, 99.84.149.182, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|99.84.149.117|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-04 15:43:31 (205 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [17]:
df_zones = spark.read.option("recursiveFileLookup", "true").option("header", "true").csv('./taxi_zone_lookup.csv')
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [20]:
df_yellow.groupBy('PULocationID').count().sort(F.asc('count')).show()

+------------+-----+
|PULocationID|count|
+------------+-----+
|         105|    1|
|           5|    1|
|          84|    1|
|         187|    3|
|         204|    4|
|         199|    4|
|         111|    4|
|         109|    4|
|           2|    5|
|         251|   12|
|          59|   14|
|         176|   14|
|         172|   14|
|         245|   14|
|         253|   15|
|          27|   16|
|         206|   17|
|          30|   18|
|         156|   21|
|         118|   22|
+------------+-----+
only showing top 20 rows


In [23]:
df_zones.filter(df_zones.LocationID.isin([105, 5, 84])).show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         5|Staten Island|       Arden Heights|   Boro Zone|
|        84|Staten Island|Eltingville/Annad...|   Boro Zone|
|       105|    Manhattan|Governor's Island...| Yellow Zone|
+----------+-------------+--------------------+------------+

